# Exercise 5: Variational Auto Encoder
## Submission date: 08\06\2026, 23:59.

Submitted by:

 **Wael Zidan, [student ID removed]

 **Tarek Diab, [student ID removed]

<font color='red'>Before submission, please make sure that all the plots and results are present as a colab notebook file</font>

## Configurations and initializations

This section loads libraries and configurations for various tasks for this course

In [ ]:
## Standard libraries
import os
import math
import numpy as np
import pandas as pd
from collections import defaultdict

## Imports for plotting
import matplotlib.pyplot as plt
plt.set_cmap('cividis')
%matplotlib inline
from matplotlib_inline.backend_inline import set_matplotlib_formats
%config InlineBackend.figure_formats = ['png']
from matplotlib.colors import to_rgb
import seaborn as sns

## Progress bar
from tqdm.notebook import tqdm

## PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim

# Torchvision
import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms

# Path to the folder where the datasets are/should be downloaded (e.g. MNIST)
DATASET_PATH = "../data"

# Path to the folder where the models are saved
CHECKPOINT_PATH = "./"

# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Fetching the device that will be used throughout this notebook
device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda:0")
print("Using device", device)

In [ ]:
# Transformations applied on each image => only make them a tensor
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Lambda(lambda x: (x * 255. + torch.rand_like(x)) / 256.)])

# Loading the training dataset. We need to split it into a training and validation part
train_dataset = MNIST(root=DATASET_PATH, train=True, transform=transform, download=True)

train_set, val_set = torch.utils.data.random_split(train_dataset, [50000, 10000],
                                                   generator=torch.Generator().manual_seed(42))

# Loading the test set
test_set = MNIST(root=DATASET_PATH, train=False, transform=transform, download=True)

# We define a set of data loaders that we can use for various purposes later.
train_loader = data.DataLoader(train_set, batch_size=128, shuffle=True, drop_last=True, pin_memory=True, num_workers=0)
val_loader = data.DataLoader(val_set, batch_size=128, shuffle=False, drop_last=False, num_workers=0)
test_loader = data.DataLoader(test_set, batch_size=128, shuffle=False, drop_last=False, num_workers=0)

print('Train size:', len(train_loader.dataset))
print('Validation size:', len(val_loader.dataset))
print('Test size:', len(test_loader.dataset))

In [ ]:
def show_imgs(imgs):
    num_imgs = imgs.shape[0] if isinstance(imgs, torch.Tensor) else len(imgs)
    nrow = min(num_imgs, 4)
    ncol = int(math.ceil(num_imgs/nrow))
    imgs = torchvision.utils.make_grid(imgs, nrow=nrow, pad_value=128)
    imgs = imgs.clamp(min=0, max=1)
    np_imgs = imgs.cpu().numpy()
    plt.figure(figsize=(1.5*nrow, 1.5*ncol))
    plt.imshow(np.transpose(np_imgs, (1,2,0)), interpolation='nearest')
    plt.axis('off')
    plt.show()

show_imgs([train_set[i][0] for i in range(8)])

## The VAE model

<font color='red'>

The encoder is three convolutions (the first two with stride 2, shrinking 28x28 down to 7x7) followed by two linear layers. Two small heads then read off the posterior: one gives the means, the other the log of the std, which we exponentiate. So each image maps to a Gaussian over the latent, with its own mean and variance per dimension.

The decoder runs the reverse path: two linear layers, three transposed convolutions back up to 28x28, and a sigmoid so every pixel sits in [0, 1]. That output is the mean of the reconstruction, and we keep a fixed std of 0.1.

The prior is a standard normal with nothing to learn. The KL term and sampling use it, the encoder is only for inference, and the decoder is used in both.

calc_elbo returns the negative ELBO: the reconstruction error plus the KL between the posterior and the prior. We draw one latent with the reparameterization trick so the gradient reaches the encoder. With a fixed decoder std the reconstruction is just a scaled squared error, and we drop its constant as the sheet allows. Both distributions are Gaussian, so the KL has the closed form from Lecture 6, which is what kl_to_standard_normal computes.

To sample we draw a latent from the prior and return the decoder mean, leaving the per-pixel noise out since the means look better.

</font>

In [ ]:
class VAE(nn.Module):

    def __init__(self, latent_dim, sigma=0.1, num_filters=32, hidden_dim=256):
        super().__init__()

        self.latent_dim = latent_dim
        self.sigma = sigma                       # fixed std of p(x|z)
        c = num_filters
        self.conv_shape = (2 * c, 7, 7)          # feature map at the encoder/decoder interface
        act = nn.GELU

        # Encoder
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, c, kernel_size=3, stride=2, padding=1),          # 28x28 -> 14x14
            act(),
            nn.Conv2d(c, 2 * c, kernel_size=3, stride=2, padding=1),      # 14x14 -> 7x7
            act(),
            nn.Conv2d(2 * c, 2 * c, kernel_size=3, stride=1, padding=1),  # 7x7
            act(),
            nn.Flatten(),
        )
        self.encoder_fc = nn.Sequential(
            nn.Linear(2 * c * 7 * 7, hidden_dim),
            act(),
            nn.Linear(hidden_dim, hidden_dim),
            act(),
        )
        # two learned linear heads: means and log-std of q(z|x)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_std = nn.Linear(hidden_dim, latent_dim)

        # Decoder
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            act(),
            nn.Linear(hidden_dim, 2 * c * 7 * 7),
            act(),
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(2 * c, 2 * c, kernel_size=3, stride=1, padding=1),                 # 7x7
            act(),
            nn.ConvTranspose2d(2 * c, c, kernel_size=3, stride=2, padding=1, output_padding=1),   # 7x7 -> 14x14
            act(),
            nn.ConvTranspose2d(c, 1, kernel_size=3, stride=2, padding=1, output_padding=1),       # 14x14 -> 28x28
            nn.Sigmoid(),                        # output in [0, 1] = the mean of p(x|z)
        )

        self.logs = defaultdict(list)


    def encode(self, x):
        """The posterior q(z|x): a Gaussian with a different mean and std per latent dimension."""
        h = self.encoder_fc(self.encoder_conv(x))
        mu = self.fc_mu(h)
        std = torch.exp(self.fc_log_std(h))      # the head gives log(std), so exp it back to a std
        return torch.distributions.Independent(torch.distributions.Normal(mu, std), 1)

    def decode(self, z):
        """The mean of p(x|z): an image in [0, 1]."""
        h = self.decoder_fc(z)
        h = h.reshape(-1, *self.conv_shape)
        return self.decoder_conv(h)

    def prior(self):
        """p(z) = N(0, I): standard isotropic Gaussian, no learned parameters."""
        p = next(self.parameters())
        return torch.distributions.Independent(
            torch.distributions.Normal(torch.zeros(self.latent_dim, device=p.device),
                                       torch.ones(self.latent_dim, device=p.device)), 1)

    @staticmethod
    def kl_to_standard_normal(q):
        """Closed-form KL( q(z|x) || p(z) ) summed over latent dims, where p(z)=N(0,I).

        Lecture 6 ("KL between Gaussians"):
            D_KL = 1/2 * sum_j ( sigma_j^2 + mu_j^2 - 1 - log sigma_j^2 ).
        """
        mu, std = q.mean, q.stddev
        return 0.5 * (std ** 2 + mu ** 2 - 1.0 - 2.0 * torch.log(std)).sum(dim=-1)


    def calc_elbo(self, x):
        """Negative ELBO averaged over the batch (up to the additive constant of log p(x|z)).

        Returns (neg_elbo, reconstruction_term, KL_term)."""
        q = self.encode(x)
        z = q.rsample()                          # reparameterization trick, so the sample stays differentiable
        x_hat = self.decode(z)

        # -log p(x|z) for a Gaussian with fixed std sigma: ||x_hat - x||^2 / (2 sigma^2) (+ const, omitted)
        rec_loss = ((x_hat - x) ** 2).flatten(start_dim=1).sum(dim=-1) / (2 * self.sigma ** 2)

        # KL( q(z|x) || p(z) ) in closed form (Lecture 6, "KL between Gaussians")
        kld = self.kl_to_standard_normal(q)

        neg_elbo = (rec_loss + kld).mean()
        return neg_elbo, rec_loss.mean(), kld.mean()

    def forward(self, x):
        return self.calc_elbo(x)

    @torch.no_grad()
    def sample(self, num_samples):
        """Generate images: z ~ p(z), return the means of p(x|z) (no per-pixel noise added)."""
        z = self.prior().sample((num_samples,))
        return self.decode(z)

    def log(self, key, value):
        self.logs[key].append(value)

    def training_step(self, batch):
        loss, rec_loss, KLD = self.calc_elbo(batch)
        # Log losses
        self.log('train_elbo', loss.item())
        self.log('train_recon', rec_loss.item())
        self.log('train_KLD', KLD.item())
        return loss

    @torch.no_grad()
    def validation_step(self, loader):
        loss_obj = []
        for imgs, _ in tqdm(loader, leave=False):
          imgs = imgs.to(device)
          loss_obj.append(self.calc_elbo(imgs))

        loss_obj = torch.tensor(loss_obj)
        loss, rec_loss, KLD = loss_obj.mean(dim=0)
        # Log losses
        self.log('val_elbo', loss.item())
        self.log('val_recon', rec_loss.item())
        self.log('val_KLD', KLD.item())

        return loss

    @torch.no_grad()
    def test_step(self, loader):
        loss_obj = []
        for imgs, _ in tqdm(loader, leave=False):
          imgs = imgs.to(device)
          loss_obj.append(self.calc_elbo(imgs))

        loss_obj = torch.tensor(loss_obj)
        loss, rec_loss, KLD = loss_obj.mean(dim=0)
        # Log losses
        self.log('test_elbo', loss.item())
        self.log('test_recon', rec_loss.item())
        self.log('test_KLD', KLD.item())

        return loss

## Model Training

In [ ]:
torch.manual_seed(42)

model = VAE(latent_dim=8)
print("Num params: {:,}".format(sum(p.numel() for p in model.parameters())))

model = model.to(device)
# torch.compile gives a nice speed-up on Colab (Linux) GPUs;
# the inductor backend is not supported on Windows, so we skip it there.
if torch.cuda.is_available() and os.name != "nt":
    model = torch.compile(model)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, 1, gamma=0.99)
epochs = 30

In [ ]:
# Training loop
for epoch in range(epochs):
    model.train()
    losses = []

    for imgs, _ in tqdm(train_loader, leave=False):
        imgs = imgs.to(device)
        optimizer.zero_grad()
        loss = model.training_step(imgs)
        loss.backward()
        optimizer.step()

    scheduler.step()
    model.validation_step(val_loader)

# Test loop
model.eval()

final_loss = model.test_step(test_loader)
print(f"Final ELBO: {final_loss:.2f}")

# Save the model
torch.save(model.state_dict(), 'vae_model.pth')

# Assignment 1: Training a VAE on MNIST

<font color='red'>

We trained for 30 epochs with Adam (learning rate 0.001, decayed a little each epoch), batch size 128, latent dimension 8, and a decoder std of 0.1.

Picking the latent size took some trial. Bigger spaces like 16 or 32 reconstruct better, but their KL is much larger, so the posteriors spread far from the prior and a random sample often lands in a gap that decodes to garbage. At dimension 8 with std 0.1 most samples come out as clean digits, so that is what we kept.

Everything below is the negative ELBO, the thing we minimise, so lower is better. The test number is printed by the training cell and again under the curves, split into its two terms.

</font>

In [ ]:
logs = model.logs
epochs_axis = np.arange(1, epochs + 1)

# training losses were logged once per batch -> average them per epoch
def per_epoch(key):
    return np.asarray(logs[key]).reshape(epochs, -1).mean(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
titles = ['negative ELBO', r'reconstruction term $-\log p(x|z)$', r'KL term $D_{KL}(q(z|x)\,\Vert\,p(z))$']
for ax, key, title in zip(axes, ['elbo', 'recon', 'KLD'], titles):
    ax.plot(epochs_axis, per_epoch('train_' + key), label='train')
    ax.plot(epochs_axis, logs['val_' + key], label='validation')
    ax.set_xlabel('epoch')
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()
plt.show()

print(f"Test set negative ELBO: {logs['test_elbo'][-1]:.2f}   "
      f"(reconstruction term: {logs['test_recon'][-1]:.2f}, KL term: {logs['test_KLD'][-1]:.2f})")

<font color='red'>

Both curves fall fast and then keep drifting down. Near the end validation flattens while training still improves a little, so there is a small gap but no real overfitting. Reconstruction keeps dropping as the decoder improves. The KL jumps early, since the posterior has to move off the prior to carry information, then settles around 32 nats.

Samples from the trained model:

</font>

In [ ]:
torch.manual_seed(0)
samples = model.sample(16)
show_imgs(samples)

<font color='red'>

Most of these read as real digits. They look a little soft next to real MNIST, the usual VAE look: the decoder mean averages over everything that could have produced that latent, so fine detail washes out.

</font>

# Assignment 2: Latent traversals

<font color='red'>

For three pairs of different digits we take a posterior sample of each image as its representation, walk in a straight line between the two latents in ten steps, and decode each step to its reconstruction mean. Under each latent row we put the matching pixel-space blend of the two images. Three pairs, two rows each, six rows in all.

For the endpoints we pick a representative image of each digit, the one the model reconstructs best among the first 20, so an unlucky encoder miss does not set the endpoint.

</font>

In [ ]:
torch.manual_seed(1)

digit_pairs = [(0, 1), (3, 8), (4, 9)]
num_steps = 10

@torch.no_grad()
def pick_representative(digit, num_candidates=20):
    """Among the first candidates of this digit in the test set, return the image whose
    posterior-mean reconstruction is the most faithful."""
    idxs = (test_set.targets == digit).nonzero()[:num_candidates, 0]
    imgs = torch.stack([test_set[int(i)][0] for i in idxs]).to(device)
    rec = model.decode(model.encode(imgs).mean)
    errs = ((rec - imgs) ** 2).flatten(start_dim=1).sum(dim=-1)
    return imgs[errs.argmin()].cpu()

@torch.no_grad()
def latent_traversal(x1, x2, num_steps=10):
    z1 = model.encode(x1[None].to(device)).sample()   # a sample from q(z|x1) = the representation
    z2 = model.encode(x2[None].to(device)).sample()
    alphas = torch.linspace(0, 1, num_steps, device=device)[:, None]
    z_path = (1 - alphas) * z1 + alphas * z2          # linear path in latent space
    return model.decode(z_path).cpu()                 # means of p(x|z^i)

def pixel_traversal(x1, x2, num_steps=10):
    alphas = torch.linspace(0, 1, num_steps)[:, None, None, None]
    return (1 - alphas) * x1[None] + alphas * x2[None]

fig, axes = plt.subplots(2 * len(digit_pairs), num_steps, figsize=(1.2 * num_steps, 1.25 * 2 * len(digit_pairs)))
for k, (d1, d2) in enumerate(digit_pairs):
    x1 = pick_representative(d1)
    x2 = pick_representative(d2)
    rows = [('latent', latent_traversal(x1, x2, num_steps)),
            ('pixels', pixel_traversal(x1, x2, num_steps))]
    for r, (name, imgs) in enumerate(rows):
        for i in range(num_steps):
            ax = axes[2 * k + r, i]
            ax.imshow(imgs[i, 0].numpy(), cmap='gray', vmin=0, vmax=1)
            ax.set_xticks([]); ax.set_yticks([])
        axes[2 * k + r, 0].set_ylabel(f'${d1}\\to{d2}$\n{name}', fontsize=10)
fig.suptitle('Latent-space vs. pixel-space traversals', y=0.99)
plt.tight_layout()
plt.show()

<font color='red'>

Along the latent path almost every step is still a believable digit: strokes bend and reconnect so the first digit slides into the second. The pixel path is just a cross-fade, both digits showing at once as faint overlays, with nothing real in between.

That is the point. The VAE lays out its latent space so a straight line between two encodings stays on the digit manifold once decoded, while a straight line in pixel space leaves it right away.

</font>

# Assignment 3: Image inpainting with the frozen VAE

<font color='red'>

The task: given the top half of an MNIST image, guess a plausible bottom half. We try three methods, all keeping the trained VAE frozen, all on the same six test images. In the figures the unknown bottom half of the observed column is filled with gray.

</font>

In [ ]:
# freeze the VAE: from here on we only use it, we never train it again
model.eval()
model.requires_grad_(False)

sigma = model.sigma
latent_dim = model.latent_dim
num_examples = 6

# the same 6 test images (different digits) for all three methods: first occurrence of each digit
first_idx = {d: int((test_set.targets == d).nonzero()[0]) for d in range(10)}
inpaint_digits = [0, 1, 3, 5, 7, 9]
x_gt = torch.stack([test_set[first_idx[d]][0] for d in inpaint_digits]).to(device)   # (6, 1, 28, 28)

# mask: 1 on the observed top half, 0 on the missing bottom half
mask = torch.zeros_like(x_gt)
mask[:, :, :14, :] = 1.0
x_obs = x_gt * mask

# for display only: show the missing half in gray
x_obs_disp = (x_obs + 0.5 * (1 - mask)).cpu()

def show_inpainting_results(x_result, method_name):
    """rows = examples, columns = [observed half | ground truth | result]"""
    fig, axes = plt.subplots(num_examples, 3, figsize=(4.6, 1.55 * num_examples))
    col_titles = ['observed half', 'ground truth', 'result']
    cols = [x_obs_disp, x_gt.cpu(), x_result.detach().cpu()]
    for r in range(num_examples):
        for c in range(3):
            ax = axes[r, c]
            ax.imshow(cols[c][r, 0].numpy(), cmap='gray', vmin=0, vmax=1)
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(col_titles[c], fontsize=10)
    fig.suptitle(method_name)
    plt.tight_layout()
    plt.show()

### (a) Iterative encode-decode with clamping

<font color='red'>

Fill the missing pixels with uniform noise, then keep running the image through the encoder and decoder, pasting the real top half back after each pass. We run 30 of these and show a few snapshots.

</font>

In [ ]:
torch.manual_seed(0)
num_iters = 30
snapshot_iters = [0, 1, 2, 3, 5, 10, 20, 30]

with torch.no_grad():
    x = mask * x_obs + (1 - mask) * torch.rand_like(x_obs)    # start the missing pixels off as uniform noise in [0, 1]
    snapshots = [x.clone()]
    for t in range(1, num_iters + 1):
        z = model.encode(x).mean                              # posterior mean of q(z | x_t)
        x_rec = model.decode(z)                               # decoder mean
        x = mask * x_obs + (1 - mask) * x_rec                 # clamp the observed pixels
        if t in snapshot_iters:
            snapshots.append(x.clone())
    x_inpaint_a = x

# snapshots: rows = examples, columns = iterations
fig, axes = plt.subplots(num_examples, len(snapshot_iters),
                         figsize=(1.25 * len(snapshot_iters), 1.3 * num_examples))
for r in range(num_examples):
    for c, t in enumerate(snapshot_iters):
        ax = axes[r, c]
        ax.imshow(snapshots[c][r, 0].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title(f'iter {t}', fontsize=9)
fig.suptitle('Method (a): iterative encode-decode with clamping -- snapshots')
plt.tight_layout()
plt.show()

show_inpainting_results(x_inpaint_a, 'Method (a): iterative encode-decode')

### (b) MAP estimate of z via gradient descent

<font color='red'>

Here we ignore the encoder and search for the latent that best explains the observed half under the prior. Starting from a random normal sample, Adam balances two things: matching the observed top half, and keeping the latent near the prior. Once it converges we decode and fill in the missing pixels.

</font>

In [ ]:
torch.manual_seed(0)
num_steps = 800

z = torch.randn(num_examples, latent_dim, device=device, requires_grad=True)   # init z ~ N(0, I)
opt = optim.Adam([z], lr=0.05)

losses_b = []
for step in range(num_steps):
    x_hat = model.decode(z)
    rec = (((x_hat - x_obs) * mask) ** 2).flatten(start_dim=1).sum(dim=-1) / (2 * sigma ** 2)
    prior_term = 0.5 * (z ** 2).sum(dim=-1)
    loss_per_image = rec + prior_term
    opt.zero_grad()
    loss_per_image.sum().backward()
    opt.step()
    losses_b.append(loss_per_image.detach().cpu())
losses_b = torch.stack(losses_b)

with torch.no_grad():
    x_inpaint_b = mask * x_obs + (1 - mask) * model.decode(z)

plt.figure(figsize=(6.5, 4))
for i, d in enumerate(inpaint_digits):
    plt.plot(losses_b[:, i], label=f'digit {d}')
plt.xlabel('Adam step'); plt.ylabel('$L(z)$'); plt.yscale('log')
plt.title('Method (b): loss curves of the MAP optimization of $z$')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

show_inpainting_results(x_inpaint_b, 'Method (b): MAP estimate of $z$')

### (c) Maximum likelihood on x via gradient descent

<font color='red'>

Now we freeze the whole VAE and optimise the missing pixels themselves, so the full image looks as likely as the model can make it. We put the free pixels through a sigmoid to keep them in [0, 1] and minimise the negative ELBO, resampling the latent each step so the gradient flows back into the pixels.

</font>

In [ ]:
torch.manual_seed(0)
num_steps = 800

y = torch.zeros_like(x_gt, requires_grad=True)    # x_free = sigmoid(y) starts at 0.5 everywhere
opt = optim.Adam([y], lr=0.05)

losses_c = []
for step in range(num_steps):
    x_free = torch.sigmoid(y)
    x_full = mask * x_obs + (1 - mask) * x_free
    q = model.encode(x_full)
    z = q.rsample()                               # reparameterised -> gradient flows into y
    x_hat = model.decode(z)
    rec = ((x_hat - x_full) ** 2).flatten(start_dim=1).sum(dim=-1) / (2 * sigma ** 2)
    kld = model.kl_to_standard_normal(q)          # same closed-form KL (Lecture 6)
    loss_per_image = rec + kld                    # negative ELBO of x(y)
    opt.zero_grad()
    loss_per_image.sum().backward()
    opt.step()
    losses_c.append(loss_per_image.detach().cpu())
losses_c = torch.stack(losses_c)

with torch.no_grad():
    x_inpaint_c = mask * x_obs + (1 - mask) * torch.sigmoid(y)

plt.figure(figsize=(6.5, 4))
for i, d in enumerate(inpaint_digits):
    plt.plot(losses_c[:, i], label=f'digit {d}')
plt.xlabel('Adam step'); plt.ylabel('$L(y) = -$ELBO$(x(y))$'); plt.yscale('log')
plt.title('Method (c): loss curves of the optimization of the missing pixels')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

show_inpainting_results(x_inpaint_c, 'Method (c): maximum likelihood on $x$')

## Comparison of the three methods

<font color='red'>

All three use the same frozen VAE and the same six images. What changes is which part of the model does the work.

Method (a) leans entirely on the encoder: each pass projects the image back onto the model's manifold, and the only thing tying it to the observation is the hard clamp. There is no prior and no objective, just a fixed-point loop. Method (b) drops the encoder and balances two pulls instead, matching the observed half and staying near the prior, the same two forces the ELBO trades off in Lecture 6. Method (c) uses all three pieces: the encoder reads the current image, the prior shows up in the KL, and the reconstruction covers the whole image including the pixels being optimised, so the bottom half is pushed toward something the model can encode and decode consistently.

They can still miss, each for a different reason. Method (a) runs the encoder on inputs it never saw in training (noise at first, half-wrong digits later), and with no objective behind it nothing stops it from locking onto the wrong digit. Method (b) has a sound goal but it is non-convex, so Adam settles into a local optimum, that mode-seeking behaviour from Lecture 6 where one latent commits to a single explanation and the top can match while the bottom belongs to another digit. Method (c) optimises a lower bound rather than the real likelihood, and even the true optimum would be the single most likely image, which for this kind of VAE is blurry rather than a typical-looking digit.

What we saw matches that. Method (a) converges in a few iterations and gives the cleanest fills, but switches identity when the top is ambiguous (the 7 became a 2, the 3 closed into an 8). Method (b) is the most principled and sharp where it works, but its loss curves settle at very different heights, which is the local minima showing up, and a couple of digits come out wrong. Method (c) reaches low values of its own objective but its fills are the blurriest, with faint smudges in the bottom half.

</font>